# Class 11 - Indexing & Filtering
### Module 3 . Week 4 . Saturday

You already know boolean masking from NumPy - `arr[arr > 5]`. Pandas extends
that exact idea to DataFrames, plus adds two formal accessors, `.loc` and
`.iloc`, that remove ambiguity about whether you mean labels or positions.

This class is about **selection** - getting exactly the rows and columns you
need, with conditions as complex as your analysis requires. This is the single
most-used skill in practical data analytics. You'll write `.loc` and boolean
filters in nearly every notebook you ever build.

We continue with the same Titanic dataset from yesterday. Today you go deeper
- selecting exactly the rows and columns your analysis needs.

## Learning Objectives

- Select data by label with `.loc[]` and by position with `.iloc[]`
- Combine multiple conditions correctly with Boolean filters `&`, `|`, `~`
- Use `.isin()`, `.between()`, `.str.contains()` for common filter patterns
- Filter with the more readable `.query()` syntax
- Build derived columns and one-hot encode a categorical column (`pd.get_dummies()`)

In [11]:
import pandas as pd
import numpy as np

# DATA_DIR = "./datasets"
# pd.set_option("display.max_columns", None)

df = pd.read_csv(r"C:\Users\PMLS\Desktop\hands-on-data-analytics-python\notebooks\03_Module3_Pandas\week_04\datasets\titanic.csv")
df.head(6)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q


In [14]:
df.loc[0:3, 'Sex':'SibSp']

,Sex,Age,SibSp
0,male,22.0,1
1,female,38.0,1
2,female,26.0,0
3,female,35.0,1


## .loc - label-based selection

`.loc[row_labels, col_labels]`. Both axes (rows and columns) are selected by their **names**, **not** their position. Slices are **inclusive on both ends** - this is the opposite of normal Python slicing.

In [7]:
df1 = pd.read_csv("datasets/patients.csv", index_col="patient_id")
df1

,name,age,gender,bp_sys,visit_date,temp
patient_id,,,,,,
P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,38.9
P002,Sara Khan,28 years,female,118.0,2024-01-11,36.7
P003,BILAL ahmed,51yo,M,NaN,2024-01-12,39.4
P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,37.1
P004,Zara Malik,45 yr old,FEMALE,132.0,2024-01-13,38.2
P005,hassan ALI,67,male,142.0,2024-01-14,34.3
P006,Nida Butt,39 years,F,NaN,2024-01-15,38.1
P007,USMAN sheikh,NaN,Female,999.0,2024-01-16,35.2
P008,Ayesha Nawaz,31 yrs,female,128.0,2024-01-17,36.7


In [ ]:
#get the age of POO4
df1.loc['P004', 'age']

'45 yr old'

In [ ]:
print("\n=== .loc — single row by label ===")
print(df1.loc['P001'])

print("\n=== .loc — multiple rows ===")
print(df1.loc[['P001', 'P003']])

print("\n=== .loc — row range (INCLUSIVE on both ends, unlike Python slicing!) ===")
print(df1.loc["P002":"P004"])    # includes S04!


=== .loc — single row by label ===
                    name     age gender  bp_sys  visit_date  temp
patient_id                                                       
P001          ahmad RAZA  34 yrs   Male   145.0  2024-01-10  38.9
P001          ahmad RAZA  34 yrs   Male   145.0  2024-01-10  37.1

=== .loc — multiple rows ===
                    name     age gender  bp_sys  visit_date  temp
patient_id                                                       
P001          ahmad RAZA  34 yrs   Male   145.0  2024-01-10  38.9
P001          ahmad RAZA  34 yrs   Male   145.0  2024-01-10  37.1
P003         BILAL ahmed    51yo      M     NaN  2024-01-12  39.4
P003         BILAL ahmed    51yo      M   158.0  2024-03-20  38.2

=== .loc — row range (INCLUSIVE on both ends, unlike Python slicing!) ===
                     name        age   gender  bp_sys  visit_date  temp
patient_id                                                             
P002            Sara Khan   28 years  female    118.0  

In [6]:
# Single row by label
df.loc[0]

PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
Name: 0, dtype: object

In [ ]:
# Multiple specific rows
df.loc[[1,3,5]]

,Name,Pclass,Sex
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,female
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,female
5,"Moran, Mr. James",3,male


In [8]:
# Row AND column selection together
df.loc[[0, 1, 2], ["Name", "Age", "Fare"]]

,Name,Age,Fare
0,"Braund, Mr. Owen Harris",22.0,7.2500
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
2,"Heikkinen, Miss. Laina",26.0,7.9250


### Boolean array directly inside .loc

In [9]:
first_class = df["Pclass"] == 1
df.loc[first_class, ["Name", "Pclass", "Fare"]].head()

,Name,Pclass,Fare
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,71.2833
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,53.1000
6,"McCarthy, Mr. Timothy J",1,51.8625
11,"Bonnell, Miss. Elizabeth",1,26.5500
23,"Sloper, Mr. William Thompson",1,35.5000


## `.iloc[row_positions, col_positions]` - position-based selection

Pure integer positions, same rules as Python lists and NumPy. Stop is **exclusive**, matching standard Python slicing.

In [10]:
df.iloc[0]               # first row by position

PassengerId                          1
Survived                             0
Pclass                               3
Name           Braund, Mr. Owen Harris
Sex                               male
Age                               22.0
SibSp                                1
Parch                                0
Ticket                       A/5 21171
Fare                              7.25
Cabin                              NaN
Embarked                             S
Name: 0, dtype: object

In [11]:
df.iloc[0:3]              # rows 0,1,2 - position 3 is excluded
# df.loc[0:3]       #???

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [12]:
df.iloc[0, 3]             # row 0, column 3 -> that passenger's Name

'Braund, Mr. Owen Harris'

In [13]:
df.iloc[:5, :4]           # first 5 rows, first 4 columns

,PassengerId,Survived,Pclass,Name
0,1,0,3,"Braund, Mr. Owen Harris"
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,3,1,3,"Heikkinen, Miss. Laina"
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)"
4,5,0,3,"Allen, Mr. William Henry"


**Difference, side by side:**
- `.loc[0:3]` would include row labelled `3` - inclusive
- `.iloc[0:3]` excludes position `3` - standard Python slicing

In [14]:
print("Rule of thumb:")
print("  .loc  ---> think 'by NAME', slices are inclusive on both ends")
print("  .iloc ---> think 'by POSITION', slices follow standard Python rules")

Rule of thumb:
  .loc  ---> think 'by NAME', slices are inclusive on both ends
  .iloc ---> think 'by POSITION', slices follow standard Python rules


`Question:` If a DataFrame has custom index labels ['a', 'b', 'c'], what is the difference between `df.loc[0]` and `df.iloc[0]`. Which one raises an error?

## Boolean filters

The NumPy pattern from Module 2, applied to DataFrame columns. **Always parenthesize each condition** when combining with `&`, `|`, `~`.

In [15]:
# Single condition
df[df["Pclass"] == 1].head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S


### Multiple conditions - parentheses are mandatory, not stylistic

In [16]:
survivors_1st = df[(df["Pclass"] == 1) & (df["Survived"] == 1)]
print(f"First-class survivors: {len(survivors_1st)}")
survivors_1st[["Name","Sex","Age"]].head()

First-class survivors: 136


,Name,Sex,Age
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0
11,"Bonnell, Miss. Elizabeth",female,58.0
23,"Sloper, Mr. William Thompson",male,28.0
31,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN


In [17]:
# OR and NOT
elderly_or_child = df[(df["Age"] > 60) | (df["Age"] < 5)]
print(f"Passengers over 60 or under 5: {len(elderly_or_child)}")

not_third_class = df[~(df["Pclass"] == 3)]
print(f"Not in 3rd class: {len(not_third_class)}")

Passengers over 60 or under 5: 62
Not in 3rd class: 400


`Question:` Why does `df[df['Age'] > 18 and df['Sex'] == 'female']` throw a ValueError, but `df[(df['Age'] > 18) & (df['Sex'] == 'female')]` works?

## .isin(), .between(), .str.contains()

In [18]:
# isin() - membership test, vectorised version of 'in'
from_cq = df[df["Embarked"].isin(["C", "Q"])]
print(f"Embarked at Cherbourg or Queenstown: {len(from_cq)}")

Embarked at Cherbourg or Queenstown: 245


In [19]:
# between() - inclusive range check
young_adults = df[df["Age"].between(20, 30)]
print(f"Passengers aged between 20-30 (inclusive): {len(young_adults)}")

Passengers aged between 20-30 (inclusive): 245


Pandas Series is a container object rather than a raw string, methods like `.contains()`, `.lower()`, or `.split()` cannot be called directly on the Series itself-.str acts as the bridge to apply those string operations element-wise
By default, `.str.contains()` returns `NaN` whenever it encounters a missing value. Passing `na=False` replaces those `NaN` results with `False`.

In [20]:
# str.contains() - text pattern matching (works like 'in' for strings)
married_women = df[df["Name"].str.contains("Mrs.", na=False)]       #case=False for (case-insensitive search) 
print(f"Passengers titled 'Mrs.': {len(married_women)}")
married_women[["Name","Age"]].head()

Passengers titled 'Mrs.': 129


,Name,Age
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0
15,"Hewlett, Mrs. (Mary D Kingcome)",55.0


In [22]:
# married_women     #--> a new DataFrame containing only the rows where "Mrs." was found

In [ ]:
## .value_counts(), .unique()

`Question:` If a column has 10 missing values (NaN), what output type does `df['Name'].str.contains('Mrs.')` return for those rows, and why does that break boolean indexing if `na=False` is omitted?

## .query() - readable filter syntax

An alternative way to filter, `using a string expression`. Many analysts prefer this for complex multi-condition filters because it reads almost like English and you can use `and`/`or` directly (unlike boolean indexing, which requires `&`/`|`)

In [19]:
# Equivalent to boolean filter, but reads as plain English
# query() lets you use 'and'/'or'/'not' (unlike boolean indexing)
df.query("Pclass == 1 and Survived == 1").shape[0]

136

In [20]:
# we can reference an external variable with @
min_fare = 50
df.query("Fare > @min_fare").shape[0]

160

## Derived columns and one-hot encoding

In [21]:
# Derived column - simple arithmetic on existing columns
df["family_size"] = df["SibSp"] + df["Parch"] + 1
df[["Name","SibSp","Parch","family_size"]].head()

,Name,SibSp,Parch,family_size
0,"Braund, Mr. Owen Harris",1,0,2
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,2
2,"Heikkinen, Miss. Laina",0,0,1
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,2
4,"Allen, Mr. William Henry",0,0,1


`Question:` Using the `FamilySize column` we just created, write a single line of code to `create` a binary column `Is_Alone` that equals 1 `if` FamilySize == 1 and 0 otherwise.

In [22]:
# Conditional derived column using np.where (from Module 2)
df["age_group"] = np.where(df["Age"] < 18, "Child",
                   np.where(df["Age"] < 60, "Adult", "Senior"))
df[["Name","Age","age_group"]].head()

,Name,Age,age_group
0,"Braund, Mr. Owen Harris",22.0,Adult
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,Adult
2,"Heikkinen, Miss. Laina",26.0,Adult
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,Adult
4,"Allen, Mr. William Henry",35.0,Adult


In [23]:
# Dropping columns
df_cleaned = df.drop(columns=["family_size"])
print(f"\nAfter dropping family_size:\n")
df_cleaned.head()


After dropping family_size:



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,age_group
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Adult
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Adult
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Adult
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Adult
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Adult


In [ ]:
# pd.get_dummies() - one-hot encoding, essential before many ML algorithms
sex_dummies = pd.get_dummies(df["Sex"], prefix="sex")       #prefix_sep=' ', drop_first=True
print(sex_dummies.head())
print(f"\nShape before: {df.shape}   after concat: {pd.concat([df, sex_dummies], axis=1).shape}")

   sex_female  sex_male
0       False      True
1        True     False
2        True     False
3        True     False
4       False      True

Shape before: (891, 14)   after concat: (891, 16)


## Practical

### Practical 1 - The missing-parentheses bug

In [41]:
try:
    bad = df[df["Age"] > 30 & df["Pclass"] == 1]
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

print()
print("Python evaluates & BEFORE > due to operator precedence.")
print("Without parentheses, this tries something nonsensical (Age>(30& df['Pclass])==1).")
print()

good = df[(df["Age"] > 30) & (df["Pclass"] == 1)]
print(f"Correct version works: {len(good)} matching rows")
print("Rule: every condition combined with &, |, ~ needs its own parentheses.")

ERROR: ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Python evaluates & BEFORE > due to operator precedence.
Without parentheses, this tries something nonsensical (Age>(30& df['Pclass])==1).

Correct version works: 125 matching rows
Rule: every condition combined with &, |, ~ needs its own parentheses.


### Practical 2 - and/or vs &/| on a Series

In [28]:
try:
    bad = df[(df["Age"] > 30) and (df["Pclass"] == 1)]
except ValueError as e:
    print(f"ERROR using 'and' on a Series: {e}")

good = df[(df["Age"] > 30) & (df["Pclass"] == 1)]
print(f"\nUsing & instead: {len(good)} rows")
print()
print("'and'/'or'/'not' are for single Python booleans.")
print("'&'/'|'/'~' are for element-wise operations on a Series.")

ERROR using 'and' on a Series: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

Using & instead: 125 rows

'and'/'or'/'not' are for single Python booleans.
'&'/'|'/'~' are for element-wise operations on a Series.


### Practical 3 - .loc vs .iloc after a filter
After filtering a DataFrame, the row labels stay the same as before. Using .iloc[0] on the filtered result gives something different from .loc[0].

In [29]:
filtered = df[df["Fare"] > 100]
print(f"filtered.iloc[0]  (position 0 of what's LEFT):")
print(filtered.iloc[0][["Name","Fare"]])

first_label = filtered.index[0]
print(f"\nfiltered.loc[{first_label}]  (the ORIGINAL row label - same row here, but not always):")
print(filtered.loc[first_label][["Name","Fare"]])

print()
print("Lesson: after filtering, ORIGINAL labels stick around.")
print(".iloc -> 'give me the Nth row of what's left'")
print(".loc  -> 'give me the row I know by its original label'")

filtered.iloc[0]  (position 0 of what's LEFT):
Name    Fortune, Mr. Charles Alexander
Fare                             263.0
Name: 27, dtype: object

filtered.loc[27]  (the ORIGINAL row label - same row here, but not always):
Name    Fortune, Mr. Charles Alexander
Fare                             263.0
Name: 27, dtype: object

Lesson: after filtering, ORIGINAL labels stick around.
.iloc -> 'give me the Nth row of what's left'
.loc  -> 'give me the row I know by its original label'


### Practical 4 - query() with column names that have spaces

In [30]:
renamed = df.rename(columns={"Pclass": "Passenger Class"})
print(renamed[["Name", "Passenger Class"]].head(2))

# query() needs backticks around spaced column names
result = renamed.query("`Passenger Class` == 1")
print(f"\nFirst class via query() with backticks: {len(result)} rows")

print()
print("Best practice: standardise column names to snake_case immediately after")
print("loading, so this friction never comes up. We formalise this in Class 12.")
print("  df.columns = df.columns.str.replace(' ', '_').str.lower()")

                                                Name  Passenger Class
0                            Braund, Mr. Owen Harris                3
1  Cumings, Mrs. John Bradley (Florence Briggs Th...                1

First class via query() with backticks: 216 rows

Best practice: standardise column names to snake_case immediately after
loading, so this friction never comes up. We formalise this in Class 12.
  df.columns = df.columns.str.replace(' ', '_').str.lower()


## Practical Exercise - Conditional Data Extraction
**Difficulty: Medium**

In [31]:
# Task 1: iloc - every other row, first 4 columns
df.iloc[::2, :4].head()

,PassengerId,Survived,Pclass,Name
0,1,0,3,"Braund, Mr. Owen Harris"
2,3,1,3,"Heikkinen, Miss. Laina"
4,5,0,3,"Allen, Mr. William Henry"
6,7,0,1,"McCarthy, Mr. Timothy J"
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)"


In [32]:
# Task 2: filter - survived AND female AND first or second class
target = df[(df["Survived"] == 1) & (df["Sex"] == "female") & (df["Pclass"].isin([1,2]))]
print(f"Surviving women in 1st/2nd class: {len(target)}")
target[["Name","Pclass","Age"]].head()

Surviving women in 1st/2nd class: 161


,Name,Pclass,Age
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,38.0
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,35.0
9,"Nasser, Mrs. Nicholas (Adele Achem)",2,14.0
11,"Bonnell, Miss. Elizabeth",1,58.0
15,"Hewlett, Mrs. (Mary D Kingcome)",2,55.0


In [33]:
# Task 3: derived fare tier column based on Fare thresholds
df["fare_tier"] = np.where(df["Fare"] >= 50, "High",
                   np.where(df["Fare"] >= 15, "Medium", "Low"))
print(df["fare_tier"].value_counts())

fare_tier
Low       457
Medium    273
High      161
Name: count, dtype: int64


In [34]:
# Task 4: get_dummies for Embarked, verify shape change
embarked_dummies = pd.get_dummies(df["Embarked"], prefix="embarked")
encoded = pd.concat([df, embarked_dummies], axis=1)
print(f"Before: {df.shape}   After: {encoded.shape}")

# Task 5: verify with query()
print(f"\nVerification - 1st class survivors via query():")
print(df.query("Pclass == 1 and Survived == 1").shape[0])

Before: (891, 15)   After: (891, 18)

Verification - 1st class survivors via query():
136


## Summary

| Concept | Key point |
|---|---|
| `.loc[]` | Label-based; slices INCLUSIVE on both ends |
| `.iloc[]` | Position-based; slices follow normal Python rules |
| `&` `\|`, `~` | Required for Combining Series conditions - never `and`/`or`/`not` |
| Parentheses | Always wrap each condition: `(cond1) & (cond2)` |
| `.isin()` | Membership test - vectorised `in` |
| `.between(lo, hi)` | Inclusive range check |
| `.query()` | String-based filtering - supports `and`/`or` directly |
| `pd.get_dummies()` | One-hot encodes a categorical column |

## Homework

Predict: what would `df['Cabin'].fillna('Unknown')` do? We formalise missing-value
handling tomorrow - try to reason through the mechanics first.

**Before Sunday:** A dataset has a column `'Salary'` stored as the string `'$45,000'`. Can you predict why `df['Salary'].mean()` would fail right now? Tomorrow we fix exactly this kind of problem.